# SME_DATA: 2019-2025 LSTM baseline

This notebook trains the same dependency-light NumPy LSTM used for the verified baseline run. It creates no extra weather features and does not impute feature values. It only orders each station by date, creates 7-observation sequences, and applies train-only numerical normalization inside the model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_BRANCH = 'agent/colab-drive-nc-preprocessing'
REPO_DIR = Path('/content/SME_DATA')
DATA_CSV = Path('/content/drive/MyDrive/SME_DATA/processed_station_features/final_train_dataset_19to25_master.csv')
OUTPUT_DIR = Path('/content/drive/MyDrive/SME_DATA/processed_station_features/lstm_baseline_19to25')

print('DATA_CSV =', DATA_CSV)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
import shutil
import subprocess

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)
], check=True)
print('Repository ready:', REPO_DIR)

In [ ]:
import pandas as pd

if not DATA_CSV.exists():
    raise FileNotFoundError(f'CSV not found: {DATA_CSV}')
check = pd.read_csv(DATA_CSV)
print('shape:', check.shape)
print('date range:', int(check.Date.min()), '~', int(check.Date.max()))
print('stations:', check.STN_ID.nunique())
print('missing TA/HM:', check[['TA', 'HM']].isna().sum().to_dict())

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    'python', str(REPO_DIR / 'scripts/train_lstm_baseline.py'),
    '--input-csv', str(DATA_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--sequence-length', '7',
    '--hidden-size', '32',
    '--epochs', '30',
    '--batch-size', '512',
    '--learning-rate', '0.002',
    '--patience', '6',
    '--seed', '42',
]
subprocess.run(command, check=True)

In [ ]:
import json
from pprint import pprint

metrics = json.loads((OUTPUT_DIR / 'lstm_metrics.json').read_text(encoding='utf-8'))
pprint(metrics['validation_metrics'])
pprint(metrics['test_metrics'])

In [ ]:
history = pd.read_csv(OUTPUT_DIR / 'lstm_training_history.csv')
predictions = pd.read_csv(OUTPUT_DIR / 'lstm_test_predictions_2025.csv')
display(history)
display(predictions.head())
print('Saved files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name, path.stat().st_size, 'bytes')